In [ ]:
from utils.evaluate import evaluate_perplexity_memmap
from mla_gpt.model.model import GPTConfig, GPT  
import torch

def make_model(use_mla=True, use_svd=True, svd_rank=64, mla_latent_dim=None):
    cfg = GPTConfig(
        block_size=1024,
        vocab_size=50304,
        n_layer=6,
        n_head=8,
        n_embd=512,
        dropout=0.0,
        bias=False,
        use_mla=use_mla,          
        use_svd=use_svd,         
        svd_rank=svd_rank,
        mla_latent_dim=mla_latent_dim,
    )
    model = GPT(cfg).to("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.is_available():
        model = model.to(dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
    return model

data_dir = "data/openwebtext" 

variants = [
    ("MLA_SVD_rank64", dict(use_mla=True, use_svd=True, svd_rank=64)),
    ("MLA_noSVD", dict(use_mla=True, use_svd=False, svd_rank=None)),
    ("Causal_baseline", dict(use_mla=False, use_svd=False, svd_rank=None)),
]

for name, args in variants:
    model = make_model(**args)
    metrics = evaluate_perplexity_memmap(
        model,
        data_dir=data_dir,
        split="val",
        block_size=1024,
        batch_size=8,
        eval_iters=50,
        amp=True,
        notes=name,
    )
    print(name, metrics.to_dict())

number of parameters: 43.09M


/Users/oktayozel/Desktop/boston university/fall2025/cs599/randomized_svd/CS599-Randomized-SVD/utils/evaluate.py:76: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(enabled=False)


FileNotFoundError: [Errno 2] No such file or directory: 'data/openwebtext/val.bin'